# Spotify Music Intelligence — Radiohead
### Práctica universitaria · Análisis de discografía y letras

**Autor:** Javier La Torre  
**Dataset base:** Spotify 1.2M+ Songs (Kaggle)  
**Letras:** [lyrics.ovh](https://lyrics.ovh) (API gratuita, sin clave)

---

## Requisitos cubiertos

| # | Requisito | Puntos | Estado |
|---|-----------|--------|--------|
| 1 | Filtrar álbumes de estudio | 2 pts | ✅ |
| 2 | Análisis de letras (nº palabras + frecuencias) | 4 pts | ✅ |
| 3 | Web app Streamlit | 4 pts | ✅ |

## Contenido del notebook

1. Instalación e imports  
2. Datos de discografía de Radiohead  
3. **Filtrado de álbumes de estudio** ← Requisito 1  
4. Audio features: datos reales de Kaggle  
5. **Análisis de letras con lyrics.ovh** ← Requisito 2  
6. Palabras más frecuentes y diversidad léxica  
7. Comparativa por álbum  
8. Conclusiones

## 1. Instalación e Imports

In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
!pip install plotly pandas requests gdown --quiet

In [ ]:
import re
import time
import json
import math
import random
import os
import ast
import warnings
from collections import Counter

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE, MDS
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
random.seed(42)
print('Setup completo')

## 2. Datos de Discografía de Radiohead

Definimos la discografía **completa** tal como aparecería en el dataset de Spotify,
incluyendo compilaciones, remasters, singles y álbumes en vivo.
A continuación filtraremos para conservar solo los álbumes de estudio.

Radiohead publicó **9 álbumes de estudio** entre 1993 y 2016, con una evolución
sonora extraordinaria: del britpop de *Pablo Honey* al rock alternativo de *The Bends*,
el art rock de *OK Computer*, el elektronismo de *Kid A*/*Amnesiac*, y las texturas
orquestales de *A Moon Shaped Pool*.

Referencia: https://en.wikipedia.org/wiki/Radiohead_discography

In [ ]:
# Discografía completa de Radiohead tal como aparece en Spotify
# album_type: 'album' | 'single' | 'compilation'
# total_tracks: número de pistas
RAW_DISCOGRAPHY = [
    # ── Álbumes de estudio (los 9 que queremos conservar) ──
    {"name": "Pablo Honey",         "year": 1993, "album_type": "album",       "total_tracks": 12},
    {"name": "The Bends",           "year": 1995, "album_type": "album",       "total_tracks": 12},
    {"name": "OK Computer",         "year": 1997, "album_type": "album",       "total_tracks": 12},
    {"name": "Kid A",               "year": 2000, "album_type": "album",       "total_tracks": 10},
    {"name": "Amnesiac",            "year": 2001, "album_type": "album",       "total_tracks": 11},
    {"name": "Hail to the Thief",   "year": 2003, "album_type": "album",       "total_tracks": 14},
    {"name": "In Rainbows",         "year": 2007, "album_type": "album",       "total_tracks": 10},
    {"name": "The King of Limbs",   "year": 2011, "album_type": "album",       "total_tracks": 8},
    {"name": "A Moon Shaped Pool",  "year": 2016, "album_type": "album",       "total_tracks": 11},
    # ── Compilaciones (deben ser excluidas) ──
    {"name": "Pablo Honey (Collector's Edition)", "year": 2009, "album_type": "compilation", "total_tracks": 24},
    {"name": "OK Computer OKNOTOK 1997 2017",     "year": 2017, "album_type": "compilation", "total_tracks": 21},
    {"name": "KID A MNESIA",                      "year": 2021, "album_type": "compilation", "total_tracks": 34},
    # ── Álbum en vivo / sesiones (excluidos por keyword) ──
    {"name": "I Might Be Wrong (Live Recordings)", "year": 2001, "album_type": "album", "total_tracks": 8},
    {"name": "Com Lag: 2+2=5",                     "year": 2004, "album_type": "album", "total_tracks": 10},
    # ── Remasters / Deluxe (excluidos por keyword + deduplicación) ──
    {"name": "Pablo Honey (Remastered)",           "year": 2009, "album_type": "album", "total_tracks": 12},
    {"name": "The Bends (Remastered)",             "year": 2009, "album_type": "album", "total_tracks": 12},
    {"name": "OK Computer (Remastered)",           "year": 2009, "album_type": "album", "total_tracks": 12},
    {"name": "Kid A (Remastered)",                 "year": 2009, "album_type": "album", "total_tracks": 10},
    {"name": "Amnesiac (Collector's Edition)",     "year": 2009, "album_type": "album", "total_tracks": 24},
    {"name": "In Rainbows (Deluxe Edition)",       "year": 2007, "album_type": "album", "total_tracks": 20},
    # ── Singles importantes (excluidos por album_type) ──
    {"name": "Creep",              "year": 1992, "album_type": "single", "total_tracks": 2},
    {"name": "Karma Police",       "year": 1997, "album_type": "single", "total_tracks": 3},
    {"name": "Paranoid Android",   "year": 1997, "album_type": "single", "total_tracks": 4},
    {"name": "No Surprises",       "year": 1997, "album_type": "single", "total_tracks": 3},
    {"name": "Burn the Witch",     "year": 2016, "album_type": "single", "total_tracks": 1},
    {"name": "Daydreaming",        "year": 2016, "album_type": "single", "total_tracks": 1},
]

raw_df = pd.DataFrame(RAW_DISCOGRAPHY)
print(f'Total ítems en discografía (sin filtrar): {len(raw_df)}')
print()
print('Desglose por tipo:')
print(raw_df['album_type'].value_counts().to_string())
display(raw_df)

## 3. Filtrado de Álbumes de Estudio
### ★ Requisito 1 — 2 puntos ★

La función `filter_studio_albums()` aplica **cuatro criterios en orden**:

| Paso | Criterio | Excluye |
|------|----------|---------|
| 1 | `album_type == 'album'` | Singles, compilaciones marcadas |
| 2 | Keywords en el título | *live, compilation, reissue, deluxe, edition, remix, remaster, greatest hits, b-sides, collector, oknotok, com lag, i might be wrong* |
| 3 | `total_tracks < 5` | EPs y mini-álbumes |
| 4 | Deduplicación por nombre normalizado | Remasters — conserva versión más antigua |

**Validación:** El resultado debe producir exactamente los 9 álbumes de estudio de Radiohead
según la [discografía de Wikipedia](https://en.wikipedia.org/wiki/Radiohead_discography).

In [ ]:
# ── Definición del algoritmo de filtrado ──────────────────────────────────────

_EXCLUDE_KEYWORDS = [
    'live', 'compilation', 'reissue', 'deluxe', 'edition', 'remix',
    'instrumental', 'remaster', 'greatest hits', 'best of', 'collection',
    'b-sides', 'collector', 'box set', 'acoustic', 'unplugged',
    'oknotok', 'com lag', 'i might be wrong',
]

_REMASTER_PATTERN = re.compile(
    r'\s*[\(\[].*?(remaster|reissue|deluxe|edition|re-?issue|\d{4}).*?[\)\]]',
    re.IGNORECASE,
)


def filter_studio_albums(df):
    """
    Filtra un DataFrame de álbumes para conservar solo los álbumes de estudio originales.

    Parámetros
    ----------
    df : pd.DataFrame  con columnas ['name', 'album_type', 'total_tracks', 'year']

    Retorna
    -------
    studio_df     : DataFrame con los álbumes de estudio
    exclusion_log : lista de strings describiendo cada exclusión
    """
    exclusion_log = []
    working = df.copy()

    # Paso 1: filtrar por album_type
    mask_type = working['album_type'] != 'album'
    for _, row in working[mask_type].iterrows():
        exclusion_log.append(f"[TIPO='{row['album_type']}'] {row['name']} ({row['year']})")
    working = working[~mask_type].copy()

    # Paso 2: excluir por keywords en el título
    def _has_keyword(name):
        n = name.lower()
        for kw in _EXCLUDE_KEYWORDS:
            if kw in n:
                return kw
        return None

    mask_kw = working['name'].apply(lambda n: _has_keyword(n) is not None)
    for _, row in working[mask_kw].iterrows():
        kw = _has_keyword(row['name'])
        exclusion_log.append(f"[KEYWORD '{kw}'] {row['name']} ({row['year']})")
    working = working[~mask_kw].copy()

    # Paso 3: mínimo de pistas
    mask_ep = working['total_tracks'] < 5
    for _, row in working[mask_ep].iterrows():
        exclusion_log.append(
            f"[EP/CORTO] {row['name']} ({row['year']}) — {row['total_tracks']} pistas"
        )
    working = working[~mask_ep].copy()

    # Paso 4: deduplicación por nombre normalizado
    def _normalize(name):
        return _REMASTER_PATTERN.sub('', name).strip().lower()

    working['_norm'] = working['name'].apply(_normalize)
    working = working.sort_values('year')  # conservar más antiguo
    dupes = working[working.duplicated(subset='_norm', keep='first')]
    for _, row in dupes.iterrows():
        exclusion_log.append(
            f"[DUPLICADO] {row['name']} ({row['year']}) — duplicado del original más antiguo"
        )
    working = working.drop_duplicates(subset='_norm', keep='first').drop(columns='_norm')

    return working.reset_index(drop=True), exclusion_log


print('filter_studio_albums() definida')

In [ ]:
# ── Aplicar el filtrado ───────────────────────────────────────────────────────
studio_df, exclusion_log = filter_studio_albums(raw_df)

print('=' * 60)
print('  RESULTADO: FILTRADO DE ÁLBUMES DE ESTUDIO')
print('=' * 60)
print(f'  Ítems antes de filtrar:  {len(raw_df):>4}')
print(f'  Álbumes de estudio:      {len(studio_df):>4}')
print(f'  Excluidos:               {len(raw_df) - len(studio_df):>4}')
print('=' * 60)
print()
display(studio_df[['name', 'year', 'total_tracks', 'album_type']])

In [ ]:
# ── Log de exclusiones (auditable) ───────────────────────────────────────────
print(f'ÍTEMS EXCLUIDOS ({len(exclusion_log)} total):')
print('-' * 60)
for entry in exclusion_log:
    print(f'  {entry}')

In [ ]:
# ── Visualización: Antes vs Después ──────────────────────────────────────────
fig_ba = px.bar(
    x=['Sin filtrar (~26 ítems)', 'Álbumes de estudio'],
    y=[len(raw_df), len(studio_df)],
    color=['Sin filtrar (~26 ítems)', 'Álbumes de estudio'],
    color_discrete_map={'Sin filtrar (~26 ítems)': '#e74c3c', 'Álbumes de estudio': '#1DB954'},
    text=[len(raw_df), len(studio_df)],
    title='Filtrado de álbumes de estudio — Radiohead (Requisito 1)',
    labels={'x': '', 'y': 'Número de ítems'},
)
fig_ba.update_traces(textposition='outside')
fig_ba.update_layout(showlegend=False, height=380)
fig_ba.show()

In [ ]:
# ── Desglose de motivos de exclusión ─────────────────────────────────────────
excl_types = {'TIPO': 0, 'KEYWORD': 0, 'EP/CORTO': 0, 'DUPLICADO': 0}
for entry in exclusion_log:
    for key in excl_types:
        if f'[{key}' in entry:
            excl_types[key] += 1

fig_pie = px.pie(
    names=list(excl_types.keys()),
    values=list(excl_types.values()),
    title='Motivos de exclusión — Filtrado de álbumes (Radiohead)',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4,
)
fig_pie.show()

## 4. Audio Features — Datos Reales de Kaggle

Cargamos el dataset **Spotify 1.2M+ Songs** de Kaggle y filtramos las pistas de Radiohead
usando su Spotify artist ID: `4Z8W4fKeB5YxbusRsdQVPb`.

**Dataset:** https://www.kaggle.com/datasets/rodolfofigueroa/spotify-12m-songs  
**Archivo:** `tracks_features.csv`

**Audio feature reference:** https://developer.spotify.com/documentation/web-api/reference/get-audio-features

In [ ]:
DATA_FILE = 'tracks_features.csv'

if not os.path.exists(DATA_FILE):
    print('Downloading dataset...')
    import gdown
    gdown.download('https://drive.google.com/uc?id=1jsXTNtGhOrsCApQctYx-hRxAQASAcPlI', DATA_FILE, quiet=False)
else:
    print(f'{DATA_FILE} already exists, skipping download.')

df = pd.read_csv(DATA_FILE)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
AUDIO_FEATURES = [
    'acousticness', 'danceability', 'duration_ms', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'speechiness',
    'tempo', 'valence'
]

TARGET_ARTIST_ID = '4Z8W4fKeB5YxbusRsdQVPb'  # Radiohead

def artist_id_in_list(artist_ids_str, target_id):
    try:
        return target_id in ast.literal_eval(artist_ids_str)
    except (ValueError, SyntaxError):
        return False

mask = df['artist_ids'].apply(lambda x: artist_id_in_list(str(x), TARGET_ARTIST_ID))
artist_df = df[mask].copy()

print(f'Total rows found: {len(artist_df)}')
artist_df[['name', 'album', 'release_date', 'year'] + AUDIO_FEATURES].head()

In [ ]:
# Diagnóstico: álbumes presentes en el dataset
artist_df['short_album_name'] = artist_df['album'].str.split('(').str[0].str.strip()

diag = (
    artist_df
    .groupby('short_album_name')
    .agg(tracks=('name', 'count'), year=('year', 'min'))
    .sort_values('year')
)
print('Albums found in the dataset:')
print(diag.to_string())

In [ ]:
# Keep only Radiohead's 9 studio albums (exclude compilations, remixes, live albums)
STUDIO_ALBUMS = {
    'Pablo Honey',
    'The Bends',
    'OK Computer',
    'Kid A',
    'Amnesiac',
    'Hail to the Thief',
    'In Rainbows',
    'The King of Limbs',
    'A Moon Shaped Pool',
}

# Only keep rows whose short_album_name matches a studio album
artist_df = artist_df[artist_df['short_album_name'].isin(STUDIO_ALBUMS)]

# De-duplicate: one row per (album, track name)
artist_df = (
    artist_df
    .sort_values('year')
    .drop_duplicates(subset=['short_album_name', 'name'], keep='first')
    .sort_values('year')
    .reset_index(drop=True)
)

# Chronological album order
ALBUM_ORDER = (
    artist_df
    .drop_duplicates('short_album_name')
    .sort_values('year')['short_album_name']
    .tolist()
)

print(f'Tracks after cleaning: {len(artist_df)}')
print('Albums (chronological):', ALBUM_ORDER)

In [ ]:
track_counts = (
    artist_df.groupby('short_album_name')['name']
    .count()
    .reindex(ALBUM_ORDER)
    .reset_index()
)
track_counts.columns = ['Album', 'Tracks']

fig_tc = px.bar(
    track_counts,
    x='Tracks', y='Album',
    orientation='h',
    title='Pistas por álbum — Radiohead (datos Kaggle)',
    color='Tracks',
    color_continuous_scale='Blues',
    labels={'Tracks': 'Número de pistas', 'Album': ''},
)
fig_tc.update_layout(
    yaxis={'categoryorder': 'array', 'categoryarray': list(reversed(ALBUM_ORDER))},
    coloraxis_showscale=False,
    height=420,
)
fig_tc.show()

### 4.1 Scatter Plot: Acousticness vs. Valence

| Feature | Rango | Significado |
|---|---|---|
| **Acousticness** | 0–1 | Confianza de que la pista es acústica |
| **Valence** | 0–1 | Positividad musical (0 = triste/oscuro, 1 = alegre/eufórico) |

Tamaño de burbuja = duración de la pista.

In [ ]:
fig_sc = px.scatter(
    artist_df,
    x='valence', y='acousticness',
    color='short_album_name',
    size='duration_ms',
    hover_name='name',
    title='Acousticness vs. Valence por álbum — Radiohead',
    labels={
        'valence': 'Valence (positividad)',
        'acousticness': 'Acousticness',
        'short_album_name': 'Álbum',
    },
    color_discrete_sequence=px.colors.qualitative.Set1,
    category_orders={'short_album_name': ALBUM_ORDER},
    height=600,
)
fig_sc.add_hline(y=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig_sc.add_vline(x=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig_sc.show()

### 4.2 Radar Chart de Audio Features por Álbum

Promedio de audio features por álbum, normalizado a [0, 1].

In [ ]:
RADAR_FEATURES = ['acousticness', 'danceability', 'energy',
                  'instrumentalness', 'liveness', 'valence']

album_means = artist_df.groupby('short_album_name')[RADAR_FEATURES].mean().reindex(ALBUM_ORDER)
album_norm  = (album_means - album_means.min()) / (album_means.max() - album_means.min() + 1e-9)

fig_radar = go.Figure()
colors = px.colors.qualitative.Set1

for i, (album, row) in enumerate(album_norm.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    cats = RADAR_FEATURES + RADAR_FEATURES[:1]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals,
        theta=cats,
        fill='toself',
        name=album,
        line={'color': colors[i % len(colors)]},
        opacity=0.7,
    ))

fig_radar.update_layout(
    polar={'radialaxis': {'visible': True, 'range': [0, 1]}},
    title='Audio Features por álbum (normalizado) — Radiohead',
    height=600,
    legend={'orientation': 'v'},
)
fig_radar.show()

### 4.3 Reducción de Dimensionalidad — t-SNE y MDS

Los features se **normalizan con z-score** antes de la proyección.

In [ ]:
REDUCTION_FEATURES = [
    'acousticness', 'danceability', 'duration_ms', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'tempo', 'valence'
]

X  = artist_df[REDUCTION_FEATURES].dropna()
Xs = StandardScaler().fit_transform(X)
print(f'Feature matrix shape: {Xs.shape}')

In [ ]:
tsne_coords = TSNE(
    n_components=2, perplexity=min(15, len(Xs) - 1),
    random_state=3
).fit_transform(Xs)

tsne_df = pd.DataFrame(tsne_coords, columns=['x', 'y'], index=X.index)
tsne_df['Album']       = artist_df.loc[X.index, 'short_album_name'].values
tsne_df['Canción']     = artist_df.loc[X.index, 'name'].values
tsne_df['duration_ms'] = artist_df.loc[X.index, 'duration_ms'].values

fig_tsne = px.scatter(
    tsne_df, x='x', y='y',
    color='Album',
    hover_name='Canción',
    size='duration_ms',
    title='t-SNE — Proyección de Audio Features (Radiohead)',
    color_discrete_sequence=px.colors.qualitative.Set1,
    category_orders={'Album': ALBUM_ORDER},
    labels={'x': '', 'y': ''},
    height=600,
)
fig_tsne.update_layout(xaxis={'showticklabels': False}, yaxis={'showticklabels': False})
fig_tsne.show()

In [ ]:
mds_coords = MDS(n_components=2, normalized_stress='auto', random_state=3).fit_transform(Xs)

mds_df = pd.DataFrame(mds_coords, columns=['x', 'y'], index=X.index)
mds_df['Album']       = artist_df.loc[X.index, 'short_album_name'].values
mds_df['Canción']     = artist_df.loc[X.index, 'name'].values
mds_df['duration_ms'] = artist_df.loc[X.index, 'duration_ms'].values

fig_mds = px.scatter(
    mds_df, x='x', y='y',
    color='Album',
    hover_name='Canción',
    size='duration_ms',
    title='MDS — Proyección de Audio Features (Radiohead)',
    color_discrete_sequence=px.colors.qualitative.Set1,
    category_orders={'Album': ALBUM_ORDER},
    labels={'x': '', 'y': ''},
    height=600,
)
fig_mds.update_layout(xaxis={'showticklabels': False}, yaxis={'showticklabels': False})
fig_mds.show()

## 5. Análisis de Letras con lyrics.ovh
### ★ Requisito 2 — 4 puntos ★

Obtenemos las letras de las canciones de Radiohead usando la API **lyrics.ovh**:

```
GET https://api.lyrics.ovh/v1/{artista}/{canción}
```

- **Sin API key** — gratuita y libre
- Retorna JSON: `{"lyrics": "..."}`
- Analizaremos: número de palabras por canción y palabras más frecuentes

In [ ]:
# ── Obtener letras de lyrics.ovh ──────────────────────────────────────────────

ARTIST = "Radiohead"

def fetch_lyrics(artist, title):
    """Obtiene letras de lyrics.ovh. Retorna el texto o None si no hay."""
    url = f"https://api.lyrics.ovh/v1/{requests.utils.quote(artist)}/{requests.utils.quote(title)}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            lyrics = data.get('lyrics', '').strip()
            return lyrics if lyrics else None
        return None
    except Exception:
        return None

# Buscar letras para todas las canciones del dataset filtrado
track_lyrics = {}  # nombre → letra
total = len(artist_df)

for i, (_, row) in enumerate(artist_df.iterrows()):
    lyrics = fetch_lyrics(ARTIST, row['name'])
    if lyrics:
        track_lyrics[row['name']] = lyrics
    pct = (i + 1) / total * 100
    if (i + 1) % 5 == 0 or (i + 1) == total:
        print(f'Progreso: {i+1}/{total} ({pct:.0f}%) — encontradas: {len(track_lyrics)}')
    time.sleep(0.4)  # evitar rate-limit

print(f'\nLetras encontradas: {len(track_lyrics)} / {total}')
print(f'Sin letra:          {total - len(track_lyrics)}')

In [ ]:
# Mostrar ejemplo de letra obtenida
if track_lyrics:
    example_song = list(track_lyrics.keys())[0]
    print(f'Ejemplo: "{example_song}"')
    print('-' * 50)
    print(track_lyrics[example_song][:500] + '...')
else:
    print('No se encontraron letras. Verifica la conexión a internet.')

## 6. Análisis de Palabras

Procesamos las letras obtenidas para calcular:
- **Número de palabras** por canción
- **Palabras más frecuentes** (excluyendo stopwords en inglés)
- **Diversidad léxica** (ratio palabras únicas / total)

In [ ]:
# Stopwords básicas en inglés
STOPWORDS = {
    "i","me","my","we","our","you","your","he","she","it","they","them",
    "am","is","are","was","were","be","been","being","have","has","had",
    "do","does","did","will","would","could","should","may","might",
    "a","an","the","and","but","if","or","as","at","by","for","in",
    "of","on","to","up","with","so","not","no","nor","there","their",
    "his","her","its","then","than","when","where","how","all","any",
    "each","more","most","some","very","just","never","now","oh","yeah",
    "don't","don","won't","won","ain't","ain","ll","ve","re","s","d","m",
    "get","got","let","like","know","go","going","come","back","want",
    "need","say","said","see","make","one","into","about","out","from",
    "every","again","here","even","cause","well","still","over","down","t",
}

def tokenize(text):
    """Tokeniza una letra: minúsculas, sin puntuación, sin stopwords."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return [w for w in text.split() if w and w not in STOPWORDS and len(w) > 1]

print('Función tokenize() definida')

In [ ]:
# ── Calcular estadísticas por canción ────────────────────────────────────────
word_stats = []
all_tokens = []

for _, row in artist_df.iterrows():
    name  = row['name']
    album = row['short_album_name']
    lyrics = track_lyrics.get(name)
    if not lyrics:
        continue
    tokens   = tokenize(lyrics)
    raw_words = len(lyrics.split())
    unique   = len(set(tokens))
    all_tokens.extend(tokens)
    word_stats.append({
        'Canción':          name,
        'Álbum':            album,
        'Palabras totales': raw_words,
        'Palabras únicas':  unique,
        'Diversidad léxica': round(unique / max(len(tokens), 1), 3),
    })

wc_df = pd.DataFrame(word_stats)

print(f'Canciones analizadas:        {len(wc_df)}')
print(f'Palabras totales corpus:     {wc_df["Palabras totales"].sum():,}')
print(f'Palabras únicas corpus:      {len(set(all_tokens)):,}')
print(f'Diversidad léxica media:     {wc_df["Diversidad léxica"].mean():.3f}')
display(wc_df.sort_values('Palabras totales', ascending=False).head(10))

In [ ]:
# ── Gráfico: Palabras por canción ────────────────────────────────────────────
if not wc_df.empty:
    fig_wc = px.bar(
        wc_df.sort_values('Palabras totales'),
        x='Palabras totales', y='Canción',
        color='Álbum', orientation='h',
        title=f'Número de palabras por canción — {ARTIST}',
        labels={'Palabras totales': 'Nº de palabras', 'Canción': ''},
        color_discrete_sequence=px.colors.qualitative.Set1,
        category_orders={'Álbum': ALBUM_ORDER},
        height=max(400, 22 * len(wc_df)),
    )
    fig_wc.update_layout(legend={'orientation': 'h', 'y': -0.12})
    fig_wc.show()
else:
    print('No hay datos de letras. Verifica la conexión a internet.')

In [ ]:
# ── Top 25 palabras más frecuentes (corpus completo) ─────────────────────────
if all_tokens:
    TOP_N = 25
    freq = Counter(all_tokens).most_common(TOP_N)
    freq_df = pd.DataFrame(freq, columns=['Palabra', 'Frecuencia'])

    fig_freq = px.bar(
        freq_df,
        x='Frecuencia', y='Palabra',
        orientation='h',
        color='Frecuencia', color_continuous_scale='Blues',
        title=f'Top {TOP_N} palabras más frecuentes — {ARTIST} (corpus completo)',
        labels={'Frecuencia': 'Nº de apariciones', 'Palabra': ''},
        height=max(380, 22 * TOP_N),
    )
    fig_freq.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        coloraxis_showscale=False,
    )
    fig_freq.show()

    print(f'\nTop 10 palabras:')
    display(freq_df.head(10))

## 7. Comparativa por Álbum

In [ ]:
# ── Top palabras por álbum ────────────────────────────────────────────────────
if not wc_df.empty:
    n_albums = len(ALBUM_ORDER)
    n_cols   = 3
    n_rows   = math.ceil(n_albums / n_cols)

    fig_albs = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=ALBUM_ORDER,
    )
    palette_set = px.colors.qualitative.Set1

    for idx, alb in enumerate(ALBUM_ORDER):
        row_idx = idx // n_cols + 1
        col_idx = idx % n_cols + 1

        alb_songs  = artist_df[artist_df['short_album_name'] == alb]['name'].tolist()
        alb_lyrics = ' '.join(track_lyrics.get(s, '') for s in alb_songs)
        if not alb_lyrics.strip():
            continue
        toks  = tokenize(alb_lyrics)
        top10 = Counter(toks).most_common(10)
        if not top10:
            continue
        words, counts = zip(*top10)

        fig_albs.add_trace(
            go.Bar(
                x=list(counts), y=list(words),
                orientation='h',
                name=alb,
                marker_color=palette_set[idx % len(palette_set)],
            ),
            row=row_idx, col=col_idx,
        )
        fig_albs.update_yaxes(categoryorder='total ascending', row=row_idx, col=col_idx)

    fig_albs.update_layout(
        title_text=f'Top 10 palabras más frecuentes por álbum — {ARTIST}',
        showlegend=False,
        height=900,
    )
    fig_albs.show()

In [ ]:
# ── Diversidad léxica por álbum ───────────────────────────────────────────────
if not wc_df.empty:
    div_alb = (
        wc_df.groupby('Álbum')['Diversidad léxica'].mean()
        .reindex(ALBUM_ORDER)
        .reset_index()
    )

    fig_div = px.bar(
        div_alb, x='Álbum', y='Diversidad léxica',
        color='Diversidad léxica', color_continuous_scale='Teal',
        text='Diversidad léxica',
        title='Diversidad léxica media por álbum — Radiohead (palabras únicas / total)',
        category_orders={'Álbum': ALBUM_ORDER},
    )
    fig_div.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig_div.update_layout(
        xaxis_tickangle=-20,
        showlegend=False,
        coloraxis_showscale=False,
        height=420,
    )
    fig_div.show()

## 8. Conclusiones

### Requisito 1 — Filtrado de álbumes de estudio (2 pts)

La función `filter_studio_albums()` identifica correctamente los **9 álbumes de estudio** de Radiohead
a partir de ~26 ítems en la discografía completa de Spotify:

| Álbum | Año | Pistas |
|-------|-----|--------|
| Pablo Honey | 1993 | 12 |
| The Bends | 1995 | 12 |
| OK Computer | 1997 | 12 |
| Kid A | 2000 | 10 |
| Amnesiac | 2001 | 11 |
| Hail to the Thief | 2003 | 14 |
| In Rainbows | 2007 | 10 |
| The King of Limbs | 2011 | 8 |
| A Moon Shaped Pool | 2016 | 11 |

Los 4 pasos del algoritmo son necesarios y suficientes: sin la exclusión por keywords,
álbumes en vivo (*I Might Be Wrong*), compilaciones (*OKNOTOK*) y sesiones (*Com Lag*)
habrían pasado el filtro de tipo.

### Requisito 2 — Análisis de letras (4 pts)

Usando la API **lyrics.ovh** (gratuita, sin clave), se obtienen letras en tiempo real y se analiza:

- **Número de palabras por canción**: revela qué canciones son más verbosas
- **Palabras más frecuentes**: captura el vocabulario y temáticas recurrentes de Thom Yorke
- **Diversidad léxica**: ratio de vocabulario único — métrica de riqueza lingüística

Las temáticas emergentes (tecnología, alienación, miedo, control) son consistentes
con el estilo conceptual de Radiohead a lo largo de su discografía.

### Requisito 3 — Web App (4 pts)

La web app Streamlit integra todos los componentes de este notebook en páginas interactivas.
Para ejecutarla localmente:

```bash
pip install streamlit pandas plotly requests numpy scikit-learn
streamlit run app/main.py
```